<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 5 — Embeddings y búsqueda semántica</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De palabras-símbolo a palabras-vector: por fin el buscador entiende significado</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Requisitos de entorno.** Este lab descarga modelos preentrenados grandes (vectores FastText en español, ~varios GB). Córranlo en una máquina con suficiente RAM/VRAM o en **Google Colab** (Entorno de ejecución → GPU). El preprocesamiento y el corpus vienen del Lab 1.


## Objetivo

Dos partes. **A)** Cargar embeddings FastText en español y explorar el espacio (vecinos, la falla agua/hídrico, analogías). **B)** Construir un buscador **semántico** sobre su corpus y compararlo, con sus métricas del Lab 3, contra TF-IDF y BM25.


## 0 · Corpus procesado del Lab 1

In [1]:
from collections import Counter
import json, math, re, unicodedata
import numpy as np

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)               # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids   = [d['id'] for d in corpus]
titulos = {d['id']: d['titulo'] for d in corpus}
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


---
## Parte A · Explorar embeddings en español

**A.1** Carguen vectores FastText en español. FastText maneja morfología y palabras fuera de vocabulario (OOV) vía n-gramas de caracteres — la razón por la que es la elección para el español.

In [2]:
!pip install fasttext

import fasttext, fasttext.util
fasttext.util.download_model('es', if_exists='ignore')
ft = fasttext.load_model('cc.es.300.bin')

def vec(palabra):
    # TODO: devolver el vector de la palabra con ft
    return ft.get_word_vector(palabra)

# TODO: imprimir la dimension del embedding y el tamano del vocabulario
print(f"Dimensión del embedding: {ft.get_dimension()}")
print(f"Tamaño del vocabulario:  {len(ft.get_words())}")

Dimensión del embedding: 300
Tamaño del vocabulario:  2000000


**A.2** Vecinos más cercanos. ¿Tienen sentido semántico?

In [3]:
# TODO: para 'sequia', 'cafe', 'chiapas', imprimir sus 5 vecinos mas cercanos
#       (ft.get_nearest_neighbors). Comenten cualquier vecino sorprendente.
for palabra in ['sequia', 'cafe', 'chiapas']:
    vecinos = ft.get_nearest_neighbors(palabra, k=5)
    print(f"\nVecinos de '{palabra}':")
    for score, vecino in vecinos:
        print(f"  {vecino:25s}  {score:.4f}")


Vecinos de 'sequia':
  sequía                     0.7464
  sequias                    0.7236
  inundacion                 0.5896
  escacez                    0.5854
  sequías                    0.5713

Vecinos de 'cafe':
  café                       0.7897
  cafes                      0.7414
  cafe.                      0.7375
  cafe-                      0.7242
  cafesito                   0.7142

Vecinos de 'chiapas':
  chiapas.                   0.7302
  oaxaca                     0.7262
  tuxtla                     0.7059
  michoacan                  0.6912
  veracruz                   0.6861


**A.3** La falla del agua, a nivel de palabra. Comprueben que el embedding sí captura el significado.

In [4]:
import numpy as np
def cos_vec(a, b):
    # TODO: similitud coseno entre dos vectores numpy
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# TODO: comparar coseno(agua, hidrico) (esperado ALTO) vs coseno(agua, jirafa) (esperado BAJO)
print(f"coseno(agua, hidrico) = {cos_vec(vec('agua'), vec('hidrico')):.4f}")  # esperado ALTO
print(f"coseno(agua, jirafa)  = {cos_vec(vec('agua'), vec('jirafa')):.4f}")   # esperado BAJO

coseno(agua, hidrico) = 0.4360
coseno(agua, jirafa)  = 0.2433


_¿Qué demuestra este resultado sobre la falla de las sesiones anteriores?_ ahora por fin ya relaciona las palabras, ya entiende que hablan de lo mismo a pesar que son distintas palabras, esto debido a que los vectores se asimilan o van en la misma direccion ya que apuntan casi al mismo lugar

**A.4** Analogías por aritmética vectorial.

In [13]:
# TODO: probar rey - hombre + mujer (ft.get_analogies) y una analogia de capitales.
#       Documenten un caso que funcione y uno que falle.
print("rey - hombre + mujer:")
for score, palabra in ft.get_analogies("rey", "hombre", "mujer"):
    print(f"  {palabra:20s}  {score:.4f}")

print("mascota - perro + alas:")
for score, palabra in ft.get_analogies("mascota", "perro", "alas"):
    print(f"  {palabra:20s}  {score:.4f}")

rey - hombre + mujer:
  reina                 0.6996
  princesa              0.6584
  reina-madre           0.5786
  monarca               0.5746
  emperatriz            0.5572
  Rey                   0.5524
  reyes                 0.5444
  hija                  0.5441
  Reina                 0.5411
  consorte              0.5356
mascota - perro + alas:
  plumas                0.5618
  alitas                0.5550
  alas.La               0.5492
  alasY                 0.5321
  alas-                 0.5296
  alas.                 0.5212
  Alas                  0.5204
  alasy                 0.5180
  alas.En               0.5161
  alas.Y                0.5145


_¿Cuándo acierta la analogía y cuándo falla? ¿Por qué?_  funciona cuando la relacion es simple, por ejemplo con rey, lo detecta de inmediato que es reina, pero con algo mas dificil como con mascota no, uno esperaria que dijiera algun ave que podria ser de mascota pero no, se pierde y busca sentido por la relacion de alas, entonces cualquier cosa relacionada a esta la da ya que no encuentra la relacion si no es sencilla y se desvia

---
## Parte B · Buscador semántico sobre su corpus

**B.1** Vector de documento = **promedio** de los vectores de sus términos. Es la forma más simple de pasar de palabra a documento (su limitación motivará Sentence-BERT en el Lab 6).

In [6]:
def vector_documento(tokens):
    # TODO: promedio de los vectores de los tokens; manejar el caso vacio
    if not tokens:
        return np.zeros(ft.get_dimension())
    return np.mean([vec(t) for t in tokens], axis=0)

# TODO: construir EMB_DOCS = {id: vector_documento(tokens)} para todo el corpus
EMB_DOCS = {d['id']: vector_documento(tokens)
            for d, tokens in zip(corpus, documentos)}

print(f"Documentos embedidos: {len(EMB_DOCS)}")
print(f"Dimensión del vector: {next(iter(EMB_DOCS.values())).shape}")

Documentos embedidos: 14
Dimensión del vector: (300,)


In [7]:
# ── preprocesar (Lab 1) ──
import spacy
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

print('Entorno listo.')
MIS_STOPWORDS = nlp.Defaults.stop_words - {'no', 'ante'}

def preprocesar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFC', texto)
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'https?://\S+', '', texto)
    texto = re.sub(r'[\U0001F300-\U0001FAFF]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    doc = nlp(texto)
    return [t.lemma_ for t in doc
            if not t.is_punct and t.text not in MIS_STOPWORDS and t.text.strip()]

# ── TF-IDF (Lab 2) ──
def tf(doc):
    c, n = Counter(doc), len(doc)
    return {t: f / n for t, f in c.items()}

def idf(corpus_docs):
    N  = len(corpus_docs)
    df = Counter(t for d in corpus_docs for t in set(d))
    return {t: np.log(N / df[t]) for t in df}

def tfidf(doc, idf_):
    return {t: w * idf_.get(t, 0) for t, w in tf(doc).items()}

def coseno_dict(v1, v2):
    dot = sum(v1[t] * v2[t] for t in v1 if t in v2)
    n1  = math.sqrt(sum(w**2 for w in v1.values()))
    n2  = math.sqrt(sum(w**2 for w in v2.values()))
    return dot / (n1 * n2) if n1 and n2 else 0.0

IDF    = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

def buscar_tfidf(consulta, k=5):
    q   = tfidf(preprocesar(consulta), IDF)
    res = [(corpus[i]['id'], corpus[i]['titulo'], coseno_dict(q, v))
           for i, v in enumerate(INDICE)]
    return sorted(res, key=lambda x: x[2], reverse=True)[:k]

# ── BM25 (Lab 3) ──
avgdl = sum(len(d) for d in documentos) / len(documentos)

def idf_bm25(corpus_docs):
    N  = len(corpus_docs)
    df = Counter(t for d in corpus_docs for t in set(d))
    return {t: math.log(1 + (N - df[t] + 0.5) / (df[t] + 0.5)) for t in df}

IDF_BM25 = idf_bm25(documentos)

def bm25_score(doc, q_tokens, k1=1.5, b=0.75):
    freqs = Counter(doc)
    dl    = len(doc)
    score = 0.0
    for t in q_tokens:
        f = freqs.get(t, 0)
        if f == 0:
            continue
        score += IDF_BM25.get(t, 0) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / avgdl))
    return score

def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    q   = preprocesar(consulta)
    res = [(corpus[i]['id'], corpus[i]['titulo'], bm25_score(doc, q, k1, b))
           for i, doc in enumerate(documentos)]
    return sorted(res, key=lambda x: x[2], reverse=True)[:k]

# ── qrels y métricas (Lab 3) ──
qrels = {
    'sequia y cultivos':                {'d04': 3, 'd02': 2},
    'problemas de agua':                {'d02': 3, 'd13': 3, 'd01': 1},
    'cafe de chiapas':                  {'d03': 3},
    'turismo en el canon del sumidero': {'d05': 3},
    'tecnologia en la universidad':     {'d07': 3, 'd14': 2},
}

def _rel(qid, doc):   return qrels[qid].get(doc, 0)
def _relevantes(qid): return {d for d, g in qrels[qid].items() if g > 0}

def ndcg_at_k(ranking, qid, k=5):
    dcg  = sum(_rel(qid, d) / math.log2(i + 1)
               for i, d in enumerate(ranking[:k], start=1))
    idcg = sum(g / math.log2(i + 1)
               for i, g in enumerate(sorted(qrels[qid].values(), reverse=True)[:k], start=1))
    return dcg / idcg if idcg else 0.0

def average_precision(ranking, qid):
    rel = _relevantes(qid)
    if not rel: return 0.0
    aciertos, suma = 0, 0.0
    for i, d in enumerate(ranking, start=1):
        if d in rel:
            aciertos += 1
            suma += aciertos / i
    return suma / len(rel)

def evaluar(buscar_fn, k=5):
    N   = len(documentos)
    acc = {'nDCG@k': 0.0, 'MAP': 0.0}
    for qid in qrels:
        ranking = [doc_id for doc_id, _, _ in buscar_fn(qid, k=N)]
        acc['nDCG@k'] += ndcg_at_k(ranking, qid, k)
        acc['MAP']    += average_precision(ranking, qid)
    n = len(qrels)
    return {m: v / n for m, v in acc.items()}

print('Funciones de Labs 1-3 listas.')

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Entorno listo.
Funciones de Labs 1-3 listas.


**B.2** Buscador semántico. Reutilicen su `preprocesar` del Lab 1 para la consulta (mismo pipeline) y rankeen por coseno.

In [8]:
# TODO: peguen su preprocesar() del Lab 1.
def buscar_semantico(consulta, k=5):
    # TODO: preprocesar la consulta -> vector_documento -> coseno contra EMB_DOCS -> top-k
    q_vec = vector_documento(preprocesar(consulta))
    res = [(d['id'], d['titulo'], cos_vec(q_vec, EMB_DOCS[d['id']]))
           for d in corpus]
    return sorted(res, key=lambda x: x[2], reverse=True)[:k]

# prueba: buscar_semantico('problemas de agua') deberia recuperar d02 (crisis hidrica)
print('Búsqueda: "problemas de agua"')
for id_, titulo, score in buscar_semantico('problemas de agua'):
    print(f'  {score:.4f}  {id_}  {titulo}')

Búsqueda: "problemas de agua"
  0.7686  d13  Restablecen servicio de agua potable
  0.6681  d02  Crisis hidrica golpea la region
  0.5515  d11  Alertan por casos de dengue
  0.5230  d04  Sequia afecta cultivos de maiz
  0.5093  d01  Lluvias provocan inundaciones en Tuxtla


**B.3** Comparación de los tres sistemas. Para 3 consultas, muestren TF-IDF (Lab 2) vs. BM25 (Lab 3) vs. semántico, lado a lado.

In [9]:
# TODO: para 3 consultas, imprimir el top-5 de TF-IDF (Lab 2), BM25 (Lab 3) y semantico, lado a lado.
#       Marquen en cuales el semantico encuentra documentos que los otros dos no.
consultas_cmp = ['problemas de agua', 'sequia y cultivos de maiz', 'cafe en chiapas']

def comparar_3(consulta, k=5):
    tf_res  = buscar_tfidf(consulta, k)
    bm_res  = buscar_bm25(consulta, k)
    sem_res = buscar_semantico(consulta, k)

    print('=' * 95)
    print(f'CONSULTA: "{consulta}"')
    print(f'{"#":<3}{"TF-IDF":<30}{"BM25":<30}{"Semántico":<30}')
    for i in range(k):
        a = f'{tf_res[i][0]}  ({tf_res[i][2]:.3f})'   if i < len(tf_res)  else ''
        b = f'{bm_res[i][0]}  ({bm_res[i][2]:.3f})'   if i < len(bm_res)  else ''
        c = f'{sem_res[i][0]}  ({sem_res[i][2]:.3f})'  if i < len(sem_res) else ''
        print(f'{i+1:<3}{a:<30}{b:<30}{c:<30}')

    solo_sem = {r[0] for r in sem_res} - {r[0] for r in tf_res} - {r[0] for r in bm_res}
    if solo_sem:
        print(f'  *** Solo el semántico encuentra: {solo_sem}')

for c in consultas_cmp:
    comparar_3(c)
    print()

CONSULTA: "problemas de agua"
#  TF-IDF                        BM25                          Semántico                     
1  d13  (0.476)                  d13  (3.258)                  d13  (0.769)                  
2  d01  (0.000)                  d01  (0.000)                  d02  (0.668)                  
3  d02  (0.000)                  d02  (0.000)                  d11  (0.551)                  
4  d03  (0.000)                  d03  (0.000)                  d04  (0.523)                  
5  d04  (0.000)                  d04  (0.000)                  d01  (0.509)                  
  *** Solo el semántico encuentra: {'d11'}

CONSULTA: "sequia y cultivos de maiz"
#  TF-IDF                        BM25                          Semántico                     
1  d04  (0.431)                  d04  (6.485)                  d04  (0.766)                  
2  d02  (0.083)                  d02  (1.677)                  d03  (0.518)                  
3  d01  (0.000)                  d01  (0.0

**B.4** Re-evaluación con sus métricas del Lab 3. ¿Mejora el nDCG en las consultas “de significado”?

In [10]:
# TODO: reusen sus qrels y su nDCG del Lab 3; calculen nDCG medio para TF-IDF, BM25 y semantico.
#       ¿En que consultas mejora mas el semantico?
def buscar_semantico_full(consulta, k=14):
    return buscar_semantico(consulta, k=k)

res_tfidf = evaluar(buscar_tfidf)
res_bm25  = evaluar(buscar_bm25)
res_sem   = evaluar(buscar_semantico_full)

print(f'{"Métrica":<10}{"TF-IDF":>10}{"BM25":>10}{"Semántico":>12}{"Mejor":>10}')
print('-' * 55)
for m in res_tfidf:
    a, b, c = res_tfidf[m], res_bm25[m], res_sem[m]
    mejor = max(('TF-IDF', a), ('BM25', b), ('Sem.', c), key=lambda x: x[1])
    print(f'{m:<10}{a:>10.3f}{b:>10.3f}{c:>12.3f}{mejor[0]:>10}')

Métrica       TF-IDF      BM25   Semántico     Mejor
-------------------------------------------------------
nDCG@k         0.931     0.931       0.996      Sem.
MAP            0.914     0.914       0.973      Sem.


_¿Mejoró el nDCG? ¿En qué tipo de consultas, y por qué?_mejoro debido a que ahora ya relaciona las palabras, entonces en consultas con palabras distintas pero que significan lo mismo o similar ya es mas acertado y esto debido a que sus vectores apuntan al mismo lugar

## Entregables — Lab 5
- [ ] Carga de FastText + exploración (vecinos, agua/hídrico, analogías) con sus salidas.
- [ ] `vector_documento`, `buscar_semantico` y la comparación de los 3 sistemas.
- [ ] Re-evaluación con las métricas del Lab 3 y análisis de en qué consultas mejora.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
